In [1]:
from transformers import AutoProcessor, AutoModelForCausalLM

processor = AutoProcessor.from_pretrained("google/functiongemma-270m-it", device_map="auto")
model = AutoModelForCausalLM.from_pretrained("google/functiongemma-270m-it", dtype="auto", device_map="auto")


/Users/sauravprateek/Documents/saurav-codes/neural-nets-codelab/saurav-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 236/236 [00:00<00:00, 624.00it/s]


In [3]:
weather_function_schema = {
    "type": "function",
    "function": {
        "name": "get_current_temperature",
        "description": "Gets the current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city name, e.g. San Francisco",
                },
            },
            "required": ["location"],
        },
    }
}

message = [
    # ESSENTIAL SYSTEM PROMPT:
    # This line activates the model's function calling logic.
    {
        "role": "developer",
        "content": "You are a model that can do function calling with the following functions"
    },
    # {
    #     "role": "user", 
    #     "content": "What's the temperature in London?"
    # },
    {
        "role": "user", 
        "content": "What's the stock price of Google?"
    }
]

inputs = processor.apply_chat_template(message, tools=[weather_function_schema], add_generation_prompt=True, return_dict=True, return_tensors="pt")

out = model.generate(**inputs.to(model.device), pad_token_id=processor.eos_token_id, max_new_tokens=128)
output = processor.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

print(output)

I cannot provide real-time stock price data. My available tools are focused on temperature and location-based requests. I cannot access or analyze financial market information.


In [21]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

gemma_270m = 'google/gemma-3-270m'
gemma_2b = 'google/gemma-2b-it'
local_path = './models/gemma_function_model_exp3'
tokenizer = AutoTokenizer.from_pretrained(gemma_270m)
model = AutoModelForCausalLM.from_pretrained(local_path, device_map="cpu")

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 25121.97it/s]


In [20]:
input_text = """
SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "calculate_shipping_cost",
    "description": "Calculate the shipping cost for a package",
    "parameters": {
        "type": "object",
        "properties": {
            "package_details": {
                "type": "object",
                "properties": {
                    "weight": {
                        "type": "number",
                        "description": "The weight of the package in pounds"
                    },
                    "dimensions": {
                        "type": "object",
                        "properties": {
                            "length": {
                                "type": "number",
                                "description": "The length of the package in inches"
                            },
                            "width": {
                                "type": "number",
                                "description": "The width of the package in inches"
                            },
                            "height": {
                                "type": "number",
                                "description": "The height of the package in inches"
                            }
                        },
                        "required": [
                            "length",
                            "width",
                            "height"
                        ]
                    }
                },
                "required": [
                    "weight",
                    "dimensions"
                ]
            },
            "origin": {
                "type": "string",
                "description": "The origin address of the package"
            },
            "destination": {
                "type": "string",
                "description": "The destination address of the package"
            }
        },
        "required": [
            "package_details",
            "origin",
            "destination"
        ]
    }
}

USER: Hey, I need to calculate the shipping cost for a package. The package weighs 5 pounds and its dimensions are 10 inches in length, 8 inches in width, and 6 inches in height. The package will be shipped from New York to Los Angeles.
ASSISTANT:
"""
input_ids = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(
    **input_ids, 
    generation_config=GenerationConfig.from_dict({"max_new_tokens": 1000}),
    stop_strings=["<|endoftext|>"],
    tokenizer=tokenizer,
).to('cpu')
print(tokenizer.decode(outputs[0]))

<bos>
SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "calculate_shipping_cost",
    "description": "Calculate the shipping cost for a package",
    "parameters": {
        "type": "object",
        "properties": {
            "package_details": {
                "type": "object",
                "properties": {
                    "weight": {
                        "type": "number",
                        "description": "The weight of the package in pounds"
                    },
                    "dimensions": {
                        "type": "object",
                        "properties": {
                            "length": {
                                "type": "number",
                                "description": "The length of the package in inches"
                            },
                            "width": {
                                "type": "number",
                                "d